In [1]:
%cd /content/drive/MyDrive/Research/OCR_LLM_API

/content/drive/MyDrive/Research/OCR_LLM_API


In [2]:
!pwd

/content/drive/MyDrive/Research/OCR_LLM_API


In [ ]:
# LLM OCR Tiếng việt thường dùng trong hóa đơn
# https://huggingface.co/5CD-AI/Vintern-1B-v2

In [3]:
!pip install -U -q transformers==4.44.2 bitsandbytes
!pip install -U -q huggingface_hub
!pip install -q flask flask-cors pyngrok flash_attn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/

In [5]:
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer
import requests

# Thư viện xử lý ảnh đầu vào


In [9]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(requests.get(image_file, stream=True).raw).convert('RGB') #Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values


# Load model và test model trên Colab


In [10]:
model_name = "5CD-AI/Vintern-1B-v2"
model = AutoModel.from_pretrained(
    model_name,
    torch_dtype = torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).eval().cuda()

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=True)
generation_config = dict(max_new_tokens=512, do_sample=False, num_beams=3, repetition_penalty=3.5)

In [13]:
test_image = 'https://media-cdn-v2.laodong.vn/Storage/NewsPortal/2022/12/7/1124909/Karaoke-2.jpg'

pixel_values = load_image(test_image, max_num=12).to(torch.bfloat16).cuda()

question = '''<image>\n Nhận diện hóa đơn trong ảnh. Chỉ trả về phần liệt kê các mặt hàng dưới dạng JSON':
[
  {
    "Tên món": "Tên món",
    "Số lượng: "Số lượng",
    "Đơn giá": "Đơn giá",
    "Thành tiền": "Thành tiền"
  },
]
'''

response, history = model.chat(tokenizer, pixel_values, question, generation_config, history=None, return_history=True)

del pixel_values
print(f'User: {question}\nAssistant: {response}')

#question = "Câu hỏi khác ......"
#response, history = model.chat(tokenizer, pixel_values, question, generation_config, history=history, return_history=True)
#print(f'User: {question}\nAssistant: {response}')

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: <image>
 Nhận diện hóa đơn trong ảnh. Chỉ trả về phần liệt kê các mặt hàng dưới dạng JSON':
[
  {
    "Tên món": "Tên món",
    "Số lượng: "Số lượng",
    "Đơn giá": "Đơn giá",
    "Thành tiền": "Thành tiền"
  },
]

Assistant: [
  {
    "Tên món": "Giờ VIP222",
    "Số lượng": "1h54'",
    "Đơn giá": "500 000",
    "Thành tiền": "950 000"
  },
  {
    "Tên món": "Suối",
    "Số lượng": "3",
    "Đơn giá": "12 000",
    "Thành tiền": "36 000"
  },
  {
    "Tên món": "Hoa quả thập cẩm",
    "Số lượng": "1",
    "Đơn giá": "140 000",
    "Thành tiền": "140 000"
  },
  {
    "Tên món": "Hoa quả Bưởi",
    "Số lượng": "2",
    "Đơn giá": "220 000",
    "Thành tiền": "440 000"
  },
  {
    "Tên món": "Hoa Quả Roi",
    "Số lượng": "1",
    "Đơn giá": "100 000",
    "Thành tiền": "100 000"
  },
  {
    "Tên món": "Ken ngoại",
    "Số lượng": "14",
    "Đơn giá": "60 000",
    "Thành tiền": "840 000"
  }
]


# Triển khai Flask và Expose ra API qua Ngrok

In [14]:
# Setup Ngrok Token
from google.colab import userdata
from flask import Flask, jsonify, request
from flask_cors import CORS
from pyngrok import ngrok

authtoken = userdata.get("ngrok_token")
ngrok.set_auth_token(authtoken)

In [16]:
# Viết code flask để expose ra API

# Initialize Flask app
app = Flask(__name__)
CORS(app)

prompt = '''<image>\n Nhận diện hóa đơn trong ảnh. Chỉ trả về phần liệt kê các mặt hàng dưới dạng CSV'''


@app.route('/ocr', methods=['POST'])
def index():
  data = request.json
  image_url = data.get('image_url', None)

  response_message = ocr_by_llm(image_url, prompt)

  return jsonify({
      "response_message": response_message
  })

def ocr_by_llm(image_url, prompt):
  # image = Image.open(requests.get(image_url, stream=True).raw)

  pixel_values = load_image(image_url, max_num=6).to(torch.bfloat16).cuda()

  response_message = model.chat(tokenizer, pixel_values, prompt, generation_config)

  del pixel_values

  print(response_message)
  return response_message

if __name__ == '__main__':
  ngrok_url = ngrok.connect(5555)
  print(ngrok_url)
  app.run(port=5555)



NgrokTunnel: "https://cca7-34-147-3-56.ngrok-free.app" -> "http://localhost:5555"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5555
INFO:werkzeug:Press CTRL+C to quit
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
INFO:werkzeug:127.0.0.1 - - [03/Apr/2025 11:25:26] "POST /ocr HTTP/1.1" 200 -


| Mặt hàng | SL | DVT | Đơn giá | Thành tiền |
|---|---|---|---|---|
| Trái cây tổng hợp | 1 | đĩa | 100.000 | 100.000 |
| NV LAM | 5.43 | Giờ | 120.000 | 651.600 |
| NV THẢO | 3.9 | Giờ | 120.000 | 468.000 |
| NV TUYỆT | 5.35 | Giờ | 120.000 | 642.000 |
| NV TRINH | 4.17 | Giờ | 120.000 | 500.400 |
| Hương dương | 2 | gói | 20.000 | 40.000 |
| Thuốc lá 555 | 4 | Bao | 60.000 | 240.000 |
| Bia gôn | 80 | chai | 20.000 | 1.600.000 |
| Khăn lạnh | 11 | cái | 3.000 | 33.000 |
| NV YÊN | 1.98 | Giờ | 120.000 | 237.600 |
| binh shisa | 1 | bình | 200.000 | 200.000 |
| cơm cháy | 4 | gói | 60.000 | 240.000 |
| NGỌC KARAOKE | 0.57 | Giờ | 120.000 | 68.400 |
| Nước suối | 4 | chai | 10.000 | 40.000 |
| Chanh muôi 360 | 3 | chai | 15.000 | 45.000 |
| Bim bim | 2 | gói | 10.000 | 45.000


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
INFO:werkzeug:127.0.0.1 - - [03/Apr/2025 11:31:19] "POST /ocr HTTP/1.1" 200 -


| Tên món ăn | DVT | SL | Ghi tham số | Thành tiền |
|---|---|---|---|---|
| Bánh mì xà lách | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Đậu phộng | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Nấm súp bơ | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Sữa chua khoai tây | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Mắm nấm | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Khế khuy viên | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Sản phẩm vi sinh viên | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Thiết bị điện tử | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Máy tính xách tay | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Máy tính bảng | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Máy tính để bàn | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Máy tính cá nhân | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |
| Máy tính đệm | 100g | 4 | 35.000 VNĐ | 298.000 VNĐ |


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
INFO:werkzeug:127.0.0.1 - - [03/Apr/2025 11:33:23] "POST /ocr HTTP/1.1" 200 -


| Tên hàng | SL | Đ.Giá | T. TIỀN |
|---|---|---|---|
| ĐÁ CHANH (LỚN) | 1 | 60.000 | 60.000 |
| TIGER BAC (CHAI) | 1 | 20.000 | 20.000 |
| TIGER (LON) | 1 | 22.000 | 22.000 |
| OLONG TEA | 1 | 15.000 | 15.000 |
| CÂU SUỐI | 1 | 12.000 | 12.000 |
| UỐNG | 1 | 129.000 | 129.000 |
